# Olist 电商数据分析

本项目从 MySQL 读取 Olist 巴西电商数据，用 Pandas 完成清洗与指标构建，
再以 Pyecharts（交互图）和 Matplotlib（静态图）输出可视化结果。

## 章节顺序

| 章节 | 内容 | 主要产出 |
| --- | --- | --- |
| 1 | 分析目标 | — |
| 2 | 连接数据库 | `RAW`：11 张源表 |
| 3 | 数据清洗 | `DATA`、`CLEANING_AUDIT` |
| 4 | 指标构建 | `ANALYSIS` |
| 5 | 可视化函数 | 图表构造函数 |
| 6 | 月度趋势 | 月度订单 / GMV / 客户数 |
| 7 | 商品品类 | Top N 品类 |
| 8 | 州域、地理与营销渠道 | 州域散点、客户分布、渠道转化 |
| 9 | 履约分析与评分归因 | 迟到率、延迟分档与评分 |
| 10 | 中文字体与静态图 | `notebooks/exports/*.png` |
| 11 | 客户 RFM 分群 | 客户分群汇总与明细 |
| 12 | 业务结论 | 关键发现汇总 |

## 关键口径

后文注释会反复引用这几个字段，先集中说明一次：

- `item_gmv`：商品销售额，只累计 `order_items.price`，**不含**运费。
- `payment_value`：订单实付金额，取自 `order_payments.payment_value`，**含**运费；
  因此整体大于 `item_gmv`（本项目数据约 1,349 万 vs 1,574 万 BRL），两者不可混用。
- `freight_value`：运费，来自 `order_items.freight_value`。
- `item_count`：商品件数，按 `order_items.order_item_id` 计数。
- 有效订单：`orders.order_status` 不属于 `PARAMS["excluded_statuses"]` 的订单。
- 客户粒度：交易数据中 `customer_id` 是「一单一客户」，同一人复购会换新值，
  因此客户维度统计统一使用 `customer_unique_id`。

## 1. 分析目标

建立一套可复用的 Olist 电商经营分析流程，回答四类问题：

1. **经营规模**——订单量与商品销售额随时间怎么变化；
2. **结构分布**——哪些商品品类、哪些州域、哪些卖家贡献主要收入；
3. **客户价值**——按 R / F / M 分层，识别高价值与流失风险人群；
4. **渠道效率**——各营销来源的线索量与成交流转率差异。

In [ ]:
from contextlib import contextmanager
from importlib.util import find_spec
from pathlib import Path
import json
import html as html_lib
import os
import re
import uuid
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import pymysql
import yaml
from dotenv import load_dotenv
from sshtunnel import SSHTunnelForwarder
from IPython.display import HTML, Markdown, clear_output, display
from pyecharts import options as opts
from pyecharts.charts import Bar, Line, Scatter, Scatter3D
from pyecharts.globals import CurrentConfig, NotebookType

CurrentConfig.NOTEBOOK_TYPE = NotebookType.JUPYTER_LAB

# 若不使用ssh连接将以下代码注释无需校验文件导包即可
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "config" / "application.yml").exists():
    raise RuntimeError("Run JupyterLab from the project root or notebooks directory.")

# Change business rules here, then rerun the analysis cells below.
PARAMS = {
    "excluded_statuses": {"canceled", "unavailable"},
    "category_top_n": 15,
    "seller_top_n": 200,
    "minimum_mql": 30,
}

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
PARAMS


## 2. SSH 与 MySQL 连接

敏感配置不写进代码：`config/application.yml` 只放 `${变量名}` 占位符，
真实值由 `config/.env` 注入（`.env` 已被 `.gitignore` 排除，不会进版本库）。

`resolve_env()` 支持两种写法：

- `${MYSQL_HOST}`：必须由 `.env` 提供，缺失即报错；
- `${MYSQL_READ_TIMEOUT:600}`：`.env` 未定义时回退到默认值 `600`。

**2.1（默认）SSH 隧道**：适用于数据库只在内网监听、需要跳板机的场景。
`ssh.enabled` 为 true 时先建立隧道再连库，连接结束后自动关闭。

**2.2（可选）直连**：数据库已对本机开放、无需跳板时使用。
启用前把 `application.yml` 的 `ssh.enabled` 改为 false，并**不要执行 2.1**——
两个单元格都会定义 `mysql_endpoint()` 并加载 `RAW`，同时执行会互相覆盖。

两个单元格最终都产出 `RAW`，即 `{表名: DataFrame}` 字典，共 11 张表。
其中 `geolocation` 在 SQL 层就按邮编预聚合（百万行降到 2000 行），避免整表拉到本地。

In [ ]:
#2.1 通过ssh连接的使用
TABLE_SQL = {
    "customers": "SELECT customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state FROM customers",
    "orders": "SELECT * FROM orders",
    "order_items": "SELECT * FROM order_items",
    "order_payments": "SELECT * FROM order_payments",
    "order_reviews": "SELECT review_id, order_id, review_score, review_creation_date, review_answer_timestamp FROM order_reviews",
    "products": "SELECT * FROM products",
    "sellers": "SELECT seller_id, seller_zip_code_prefix, seller_city, seller_state FROM sellers",
    "translation": "SELECT * FROM product_category_translation",
    "mql": "SELECT * FROM marketing_qualified_leads",
    "closed_deals": "SELECT * FROM closed_deals",
    "geolocation": """
        SELECT geolocation_zip_code_prefix,
               AVG(geolocation_lat) AS latitude,
               AVG(geolocation_lng) AS longitude,
               MAX(geolocation_state) AS state,
               COUNT(*) AS records
        FROM geolocation
        GROUP BY geolocation_zip_code_prefix
        ORDER BY records DESC
        LIMIT 2000
    """,
}

ENV_PATTERN = re.compile(r"^\$\{([A-Za-z_][A-Za-z0-9_]*)(?::([^}]*))?\}$")

def resolve_env(value):
    """解析 application.yml 中的 ${VAR} 占位符，支持 ${VAR:默认值} 写法。"""
    if isinstance(value, dict):
        return {key: resolve_env(item) for key, item in value.items()}
    if isinstance(value, list):
        return [resolve_env(item) for item in value]
    if not isinstance(value, str):
        return value
    match = ENV_PATTERN.fullmatch(value)
    if not match:
        return value
    name, default = match.group(1), match.group(2)
    result = os.getenv(name)
    if result is None:
        result = default
    if result is None:
        raise ValueError(f"请在 config/.env 中填写 {name}，或在 application.yml 中为其提供默认值。")
    return result

def as_bool(value):
    return str(value).strip().lower() in {"1", "true", "yes", "on"}

def connection_config():
    env_path = ROOT / "config" / ".env"
    yaml_path = ROOT / "config" / "application.yml"
    if not env_path.is_file():
        raise FileNotFoundError("缺少 config/.env 配置文件。")

    load_dotenv(env_path, override=True)
    with yaml_path.open(encoding="utf-8") as source:
        config = resolve_env(yaml.safe_load(source))

    mysql = config["database"]["mysql"]
    if not str(mysql.get("password", "")).strip():
        raise ValueError("请在 config/.env 的 MYSQL_PASSWORD 中填写 MySQL 密码。")

    ssh = config.get("ssh", {})
    if as_bool(ssh.get("enabled", False)):
        for field in ("host", "user", "private_key"):
            if not str(ssh.get(field, "")).strip():
                raise ValueError(f"请在 config/.env 中填写 SSH 配置：{field}。")
    return config

def mysql_config(config):
    mysql = config["database"]["mysql"]
    return {
        "host": mysql["host"],
        "port": int(mysql["port"]),
        "user": mysql["user"],
        "password": mysql["password"],
        "database": mysql["database"],
        "charset": mysql.get("charset", "utf8mb4"),
        "cursorclass": pymysql.cursors.DictCursor,
        "connect_timeout": int(mysql.get("connect_timeout", 15)),
        "read_timeout": int(mysql.get("read_timeout", 600)),
    }

@contextmanager
def mysql_endpoint():
    config = connection_config()
    mysql = mysql_config(config)
    ssh = config.get("ssh", {})

    if not as_bool(ssh.get("enabled", False)):
        yield mysql
        return

    private_key = Path(ssh["private_key"]).expanduser()
    if not private_key.is_file():
        raise FileNotFoundError(f"SSH private key not found: {private_key}")

    tunnel_options = {
        "ssh_address_or_host": (ssh["host"], int(ssh.get("port", 22))),
        "ssh_username": ssh["user"],
        "ssh_pkey": str(private_key),
        "ssh_private_key_password": ssh.get("private_key_password") or None,
        "remote_bind_address": (
            ssh.get("remote_host", "127.0.0.1"),
            int(ssh.get("remote_port", 3306)),
        ),
        "local_bind_address": ("127.0.0.1", 0),
    }
    if ssh.get("host_key"):
        tunnel_options["ssh_host_key"] = ssh["host_key"]

    tunnel = SSHTunnelForwarder(**tunnel_options)
    tunnel.start()
    try:
        mysql["host"] = "127.0.0.1"
        mysql["port"] = tunnel.local_bind_port
        yield mysql
    finally:
        tunnel.stop()

def load_mysql_data():
    frames = {}
    with mysql_endpoint() as config:
        print(f"Reading MySQL through {config['host']}:{config['port']}/{config['database']} as {config['user']}")
        with pymysql.connect(**config) as connection:
            with connection.cursor() as cursor:
                for name, sql in TABLE_SQL.items():
                    cursor.execute(sql)
                    frames[name] = pd.DataFrame(cursor.fetchall())
                    print(f"{name:16s} {len(frames[name]):>10,} rows")
    return frames

# Run once per session. Parameter changes below reuse RAW without querying MySQL again.
RAW = load_mysql_data()

In [ ]:
# =====================================================================
# 2.2 【可选，备用方案】不使用 SSH，直接连接 MySQL
# ---------------------------------------------------------------------
# 默认请使用上面的 2.1（SSH 隧道）方式。仅当数据库已对本机开放、
# 无需 SSH 跳板时，才启用本单元格作为替代。
#
# 启用步骤：
#   1. 不要执行 2.1，直接执行本单元格；
#   2. 把 application.yml 中 ssh.enabled 设为 false；
#   3. 连接信息仍从 config/.env 注入（本单元格不写任何明文密码）。
#
# 注意：本单元格会覆盖 mysql_endpoint 与 RAW，请勿与 2.1 同时执行。
# =====================================================================

import re
import os
from contextlib import contextmanager
import pandas as pd
import pymysql
from pymysql.cursors import DictCursor

# TABLE_SQL 已在 2.1 单元格定义，此处不再重复，避免两处 SQL 定义不一致。

# =====================【2】直连配置：仍从 config/.env + application.yml 注入 =====================
# 请勿在此填写明文密码。连接信息统一维护在 config/.env。
# 启用前先把 application.yml 的 ssh.enabled 改为 false。
MYSQL_CONFIG = {**mysql_config(connection_config()), "cursorclass": DictCursor}

# =====================【3】直连无需 SSH 隧道，直接复用已解析的连接配置 =====================
@contextmanager
def mysql_endpoint():
    yield MYSQL_CONFIG

# =====================【4】直接复用 2.1 的加载逻辑 =====================
def load_mysql_data():
    frames = {}
    with mysql_endpoint() as config:
        print(f"Reading MySQL through {config['host']}:{config['port']}/{config['database']} as {config['user']}")
        with pymysql.connect(**config) as connection:
            with connection.cursor() as cursor:
                for name, sql in TABLE_SQL.items():
                    cursor.execute(sql)
                    frames[name] = pd.DataFrame(cursor.fetchall())
                    print(f"{name:16s} {len(frames[name]):>10,} rows")
    return frames

# 一次性加载所有数据
RAW = load_mysql_data()


## 3. 数据清洗

`clean_all()` 逐表调用 `clean_frame()`，转换规则由下面三张映射表驱动：

| 映射表 | 作用 | 主要字段 |
| --- | --- | --- |
| `DATE_COLUMNS` | `pd.to_datetime(..., errors="coerce")` 解析日期 | `order_purchase_timestamp`、`order_delivered_customer_date`、`order_estimated_delivery_date`、`shipping_limit_date`、`won_date` |
| `NUMERIC_COLUMNS` | `pd.to_numeric(..., errors="coerce")` 解析数值 | `price`、`freight_value`、`payment_value`、`review_score`、`product_weight_g`、`latitude`、`longitude` |
| `ZIP_COLUMNS` | 邮编统一为 5 位字符串 | `customer_zip_code_prefix`、`seller_zip_code_prefix`、`geolocation_zip_code_prefix` |

具体处理：

- 文本列 `strip()` 后把空串转为 `pd.NA`，统一缺失值表示；
- 邮编去掉可能的 `.0` 后缀再 `zfill(5)`，避免 `01037` 退化成 `1037`；
- 解析失败的日期不抛异常而是记为 `NaT`，并把条数累加到 `invalid_dates`。

产出两个对象：

- `DATA`：清洗后的 11 张表，后续分析全部基于它；
- `CLEANING_AUDIT`：每张表的行数、无效日期数、缺失单元格数，用于数据质量审计。

In [ ]:
DATE_COLUMNS = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
    "mql": ["first_contact_date"],
    "closed_deals": ["won_date"],
}
NUMERIC_COLUMNS = {
    "order_items": ["order_item_id", "price", "freight_value"],
    "order_payments": ["payment_sequential", "payment_installments", "payment_value"],
    "order_reviews": ["review_score"],
    "products": ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"],
    "closed_deals": ["declared_product_catalog_size", "declared_monthly_revenue"],
    "geolocation": ["latitude", "longitude", "records"],
}
ZIP_COLUMNS = {
    "customers": "customer_zip_code_prefix",
    "sellers": "seller_zip_code_prefix",
    "geolocation": "geolocation_zip_code_prefix",
}

def clean_frame(name, frame):
    result = frame.copy()
    audit = {"table": name, "rows": len(result), "invalid_dates": 0}
    for column in result.select_dtypes(include="object"):
        result[column] = result[column].astype("string").str.strip().replace("", pd.NA)
    zip_column = ZIP_COLUMNS.get(name)
    if zip_column in result:
        result[zip_column] = result[zip_column].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(5)
    for column in NUMERIC_COLUMNS.get(name, []):
        if column in result:
            result[column] = pd.to_numeric(result[column], errors="coerce")
    for column in DATE_COLUMNS.get(name, []):
        if column in result:
            source_not_null = result[column].notna()
            parsed = pd.to_datetime(result[column], errors="coerce")
            audit["invalid_dates"] += int((source_not_null & parsed.isna()).sum())
            result[column] = parsed
    audit["null_cells"] = int(result.isna().sum().sum())
    return result, audit

def clean_all(frames):
    cleaned, audit = {}, []
    for name, frame in frames.items():
        cleaned[name], stats = clean_frame(name, frame)
        audit.append(stats)
    return cleaned, pd.DataFrame(audit)

DATA, CLEANING_AUDIT = clean_all(RAW)
CLEANING_AUDIT


## 4. 指标构建

`build_analysis(DATA, PARAMS)` 是分析核心，产出一个 `ANALYSIS` 字典。

### 第一步：确定有效订单

过滤掉 `order_status` 属于 `PARAMS["excluded_statuses"]`（默认 `canceled`、`unavailable`）
的订单，得到 98,207 笔有效订单，后续统计都限定在这个集合内。

### 第二步：构建订单级事实表 `ANALYSIS["orders"]`

以订单为主体，把明细、支付、评价各自按 `order_id` 聚合后合并进来，一行一单：

| 派生字段 | 来源 | 说明 |
| --- | --- | --- |
| `item_gmv` | `order_items.price` 按单求和 | 商品销售额，不含运费 |
| `freight_value` | `order_items.freight_value` 按单求和 | 运费 |
| `item_count` | `order_items.order_item_id` 计数 | 商品件数 |
| `payment_value` | `order_payments.payment_value` 按单求和 | 实付金额，含运费 |
| `review_score` | `order_reviews.review_score` 按单取均值 | 一单可能对应多条评价 |
| `order_month` | `order_purchase_timestamp` 转 `YYYY-MM` | 月度聚合的键 |
| `on_time` | 实际送达日期 ≤ 预计送达日期 | 原始比较结果 |

> `on_time` 需注意：未送达订单的 `order_delivered_customer_date` 为 `NaT`，
> `NaT` 参与比较得到 False，因此**不能**直接用 `on_time.mean()` 当准时率。
> KPI 中用 `comparable` 子集（两个日期都非空）单独计算，避免把未送达误判为「不准时」。

合并时用 `validate=` 显式声明连接基数（`many_to_one` / `one_to_one`）：
源数据一旦出现重复键会直接抛错，而不是静默放大行数。

### 第三步：派生各分析主题表

以下键都挂在 `ANALYSIS` 下：

| 键 | 粒度 | 关键字段 | 用途 |
| --- | --- | --- | --- |
| `orders` | 订单 | `item_gmv`、`payment_value`、`on_time` | 订单级事实表 |
| `items` | 订单明细 | `price`、`freight_value`、`seller_id` | 品类与卖家分析 |
| `monthly` | 月 | `valid_orders`、`item_gmv`、`unique_customers`、`aov` | 月度趋势 |
| `category` | 商品品类 | `category`、`item_gmv`、`item_count`、`orders` | Top N 品类 |
| `state` | 州 | `customer_state`、`valid_orders`、`item_gmv` | 州域分布 |
| `seller` | 卖家 | `valid_orders`、`item_gmv`、`freight_share` | 卖家表现 |
| `marketing` | 营销来源 | `origin`、`mql_count`、`closed_deals`、`conversion_rate` | 渠道转化 |
| `rfm` | 客户 | `recency`、`frequency`、`monetary`、`r_score`、`segment` | 客户分层 |
| `rfm_summary` | 分群 | `customers`、`revenue`、`customer_share` | 分群汇总 |
| `geo_points` | 邮编区 | `latitude`、`longitude`、`customers`、`item_gmv` | 地理分布图 |
| `kpis` | — | 七个总览指标 | 见下 |

### `ANALYSIS["kpis"]` 的七个指标

`valid_orders`（有效订单数）、`item_gmv`、`payment_value`、
`aov`（客单价 = `item_gmv` ÷ 有效订单数）、
`repeat_rate`（复购率：同一 `customer_unique_id` 下单 ≥ 2 次的比例）、
`on_time_rate`（准时率：仅计入双日期齐全的订单）、
`mql_conversion`（线索转化率 = 成交线索 ÷ 营销合格线索）。

调整口径只需改开头的 `PARAMS` 后重跑本单元格，`RAW` 无需重新查询数据库。

In [ ]:
import numpy as np
def build_analysis(data, params):
    orders = data["orders"]
    valid = orders[~orders["order_status"].isin(params["excluded_statuses"])].copy()
    valid_ids = set(valid["order_id"])
    items = data["order_items"][data["order_items"]["order_id"].isin(valid_ids)].copy()

    item_order = items.groupby("order_id", as_index=False).agg(
        item_gmv=("price", "sum"), freight_value=("freight_value", "sum"), item_count=("order_item_id", "count")
    )
    payment_order = data["order_payments"].groupby("order_id", as_index=False).agg(payment_value=("payment_value", "sum"))
    review_order = data["order_reviews"].groupby("order_id", as_index=False).agg(review_score=("review_score", "mean"))

    order_fact = (valid
        .merge(data["customers"], on="customer_id", how="left", validate="many_to_one")
        .merge(item_order, on="order_id", how="left", validate="one_to_one")
        .merge(payment_order, on="order_id", how="left", validate="one_to_one")
        .merge(review_order, on="order_id", how="left", validate="one_to_one"))
    for column in ["item_gmv", "freight_value", "item_count", "payment_value"]:
        order_fact[column] = order_fact[column].fillna(0)
    order_fact["order_month"] = order_fact["order_purchase_timestamp"].dt.to_period("M").astype("string")
    # 按天比较：预计送达日期的时间部分恒为 00:00:00，若按时间戳严格比较，
    # 预计当天白天送达的订单会被误判为迟到。此处与 is_late 统一为按天口径。
    order_fact["on_time"] = (order_fact["order_delivered_customer_date"].dt.normalize()
                             <= order_fact["order_estimated_delivery_date"].dt.normalize())

    monthly = order_fact.groupby("order_month", as_index=False).agg(
        valid_orders=("order_id", "nunique"), item_gmv=("item_gmv", "sum"),
        payment_value=("payment_value", "sum"), unique_customers=("customer_unique_id", "nunique")
    )
    monthly["aov"] = monthly["item_gmv"] / monthly["valid_orders"]

    category = (items
        .merge(data["products"][["product_id", "product_category_name"]], on="product_id", how="left", validate="many_to_one")
        .merge(data["translation"], on="product_category_name", how="left", validate="many_to_one")
        .groupby("product_category_name_english", dropna=False, as_index=False)
        .agg(item_gmv=("price", "sum"), item_count=("order_item_id", "count"), orders=("order_id", "nunique")))
    category["category"] = category["product_category_name_english"].fillna("unknown")

    state = order_fact.groupby("customer_state", dropna=False, as_index=False).agg(
        valid_orders=("order_id", "nunique"), item_gmv=("item_gmv", "sum"),
        unique_customers=("customer_unique_id", "nunique"), payment_value=("payment_value", "sum")
    )

    seller = items.groupby("seller_id", as_index=False).agg(
        valid_orders=("order_id", "nunique"), item_count=("order_item_id", "count"),
        item_gmv=("price", "sum"), freight_value=("freight_value", "sum"))
    seller["freight_share"] = seller["freight_value"] / seller["item_gmv"].replace(0, pd.NA)

    mql = data["mql"].copy()
    closed_ids = set(data["closed_deals"]["mql_id"].dropna())
    mql["closed"] = mql["mql_id"].isin(closed_ids)
    marketing = mql.groupby("origin", dropna=False, as_index=False).agg(
        mql_count=("mql_id", "nunique"), closed_deals=("closed", "sum"))
    marketing["origin"] = marketing["origin"].fillna("unknown")
    marketing["conversion_rate"] = marketing["closed_deals"] / marketing["mql_count"]

    # ---- 履约分析：迟到率、迟到对评分的影响、同州/跨州对比 ----
    timely = order_fact.dropna(subset=["order_delivered_customer_date", "order_estimated_delivery_date"]).copy()
    timely["days_late"] = (timely["order_delivered_customer_date"]
                           - timely["order_estimated_delivery_date"]).dt.days.clip(lower=0)
    timely["delivery_days"] = (timely["order_delivered_customer_date"]
                               - timely["order_purchase_timestamp"]).dt.days
    timely["is_late"] = timely["days_late"] > 0
    # 新增派生列统一挂到 order_fact 上；后续单元格（含前端展示）直接引用，
    # 避免各处重复计算导致口径不一致。
    order_fact = order_fact.merge(timely[["order_id", "days_late", "delivery_days", "is_late"]],
                                  on="order_id", how="left", validate="one_to_one")

    delivery_state = timely.groupby("customer_state", dropna=False, as_index=False).agg(
        orders=("order_id", "nunique"), late_orders=("is_late", "sum"),
        avg_delivery_days=("delivery_days", "mean"),
    )
    delivery_state["late_rate"] = delivery_state["late_orders"] / delivery_state["orders"]

    seller_state = data["sellers"][["seller_id", "seller_state"]]
    same_state = (timely.merge(items[["order_id", "seller_id"]], on="order_id", how="left")
                        .merge(seller_state, on="seller_id", how="left"))
    # 注意：清洗后州字段是 pandas 可空字符串（StringDtype），未匹配到卖家时为 pd.NA。
    # 直接比较会产生 NA，而 np.where 收到 NA 会抛 "boolean value of NA is ambiguous"，
    # 因此先把 NA 显式填成占位值，保证结果只有「同州 / 跨州」两类。
    same_state["lane"] = np.where(
        same_state["customer_state"].fillna("未知") == same_state["seller_state"].fillna("未知"),
        "同州", "跨州")
    delivery_lane = same_state.groupby("lane", dropna=False, as_index=False).agg(
        lines=("order_id", "size"), late_orders=("is_late", "sum"),
        avg_freight=("freight_value", "mean"),
    )
    delivery_lane["late_rate"] = delivery_lane["late_orders"] / delivery_lane["lines"]

    delivery_buckets = (timely.assign(bucket=pd.cut(
            timely["days_late"], bins=[-1, 0, 3, 7, 14, 10 ** 6],
            labels=["准时/提前", "迟到1-3天", "迟到4-7天", "迟到8-14天", "迟到15天以上"]))
        .groupby("bucket", observed=True, as_index=False)
        .agg(orders=("order_id", "nunique"), avg_score=("review_score", "mean"),
             bad_review_rate=("review_score", lambda s: (s <= 2).mean())))

    customer_orders = order_fact.dropna(subset=["customer_unique_id"]).copy()
    snapshot_date = customer_orders["order_purchase_timestamp"].max() + pd.Timedelta(days=1)
    rfm = customer_orders.groupby("customer_unique_id", as_index=False).agg(
        last_purchase_date=("order_purchase_timestamp", "max"),
        frequency=("order_id", "nunique"),
        monetary=("payment_value", "sum"),
    )
    rfm["recency"] = (snapshot_date - rfm["last_purchase_date"]).dt.days

    geo_points = (order_fact.dropna(subset=["customer_zip_code_prefix"])
        .groupby("customer_zip_code_prefix", as_index=False)
        .agg(customers=("customer_unique_id", "nunique"), item_gmv=("item_gmv", "sum"))
        .merge(data["geolocation"], left_on="customer_zip_code_prefix",
               right_on="geolocation_zip_code_prefix", how="inner", validate="one_to_one"))
    geo_points["zip_prefix"] = geo_points["geolocation_zip_code_prefix"]

    def quintile_score(values, reverse=False):
        scores = pd.qcut(values.rank(method="first"), q=5, labels=False) + 1
        return (6 - scores if reverse else scores).astype(int)

    rfm["r_score"] = quintile_score(rfm["recency"], reverse=True)
    rfm["f_score"] = quintile_score(rfm["frequency"])
    rfm["m_score"] = quintile_score(rfm["monetary"])
    rfm["rfm_score"] = rfm["r_score"].astype(str) + rfm["f_score"].astype(str) + rfm["m_score"].astype(str)
    rfm["segment"] = "Others"
    rfm.loc[(rfm["r_score"] >= 4) & (rfm["f_score"] >= 4) & (rfm["m_score"] >= 4), "segment"] = "Champions"
    rfm.loc[(rfm["r_score"] >= 3) & (rfm["f_score"] >= 4) & (rfm["segment"] == "Others"), "segment"] = "Loyal Customers"
    rfm.loc[(rfm["m_score"] >= 4) & (rfm["segment"] == "Others"), "segment"] = "Big Spenders"
    rfm.loc[(rfm["r_score"] >= 4) & (rfm["segment"] == "Others"), "segment"] = "Recent Customers"
    rfm.loc[(rfm["r_score"] <= 2) & (rfm["f_score"] >= 3) & (rfm["segment"] == "Others"), "segment"] = "At Risk"
    segment_order = ["Champions", "Loyal Customers", "Big Spenders", "Recent Customers", "At Risk", "Others"]
    rfm_summary = rfm.groupby("segment", as_index=False).agg(
        customers=("customer_unique_id", "nunique"), avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"), revenue=("monetary", "sum"),
    )
    rfm_summary["segment"] = pd.Categorical(rfm_summary["segment"], categories=segment_order, ordered=True)
    rfm_summary = rfm_summary.sort_values("segment").reset_index(drop=True)
    rfm_summary["customer_share"] = rfm_summary["customers"] / len(rfm)
    rfm_summary["revenue_share"] = rfm_summary["revenue"] / rfm["monetary"].sum()

    repeat = order_fact.groupby("customer_unique_id")["order_id"].nunique()
    comparable = order_fact.dropna(subset=["order_delivered_customer_date", "order_estimated_delivery_date"])
    kpis = pd.Series({
        "valid_orders": order_fact["order_id"].nunique(),
        "item_gmv": order_fact["item_gmv"].sum(),
        "payment_value": order_fact["payment_value"].sum(),
        "aov": order_fact["item_gmv"].sum() / order_fact["order_id"].nunique(),
        "repeat_rate": (repeat >= 2).mean(),
        "on_time_rate": comparable["on_time"].mean(),
        "late_rate": timely["is_late"].mean(),
        "avg_delivery_days": timely["delivery_days"].mean(),
        "late_review_drop": (timely.loc[~timely["is_late"], "review_score"].mean()
                              - timely.loc[timely["is_late"], "review_score"].mean()),
        "mql_conversion": mql["closed"].mean(),
    })
    return {"orders": order_fact, "items": items, "monthly": monthly, "category": category,
            "state": state, "seller": seller, "marketing": marketing,
            "rfm": rfm, "rfm_summary": rfm_summary, "kpis": kpis,
            "geo_points": geo_points, "geolocation": data["geolocation"],
            "delivery_state": delivery_state, "delivery_lane": delivery_lane,
            "delivery_buckets": delivery_buckets}

ANALYSIS = build_analysis(DATA, PARAMS)
ANALYSIS["kpis"].to_frame("value")

## 5. 可视化函数

本单元格只定义函数，不产生输出。除业务图表外还包含两项基础设施。

**离线图表渲染**：Pyecharts 默认从 CDN 加载 `echarts.min.js`。
`_read_echarts_runtime()` 改为从已安装的 `panel` 包内读取 ECharts 与 echarts-gl，
再由 `display_pyechart()` 把图表 HTML 内联进 `iframe` 输出。
断网也能正常显示，代价是每个图表内联约 1 MB 运行库。
`echarts-gl` 仅在图表声明依赖时注入，二维图不会加载它。

**控件复用**：`replace_widget_observers()` 在重跑单元格时先注销上一次的回调，
避免交互控件叠加多个监听器而重复触发。

| 函数 | 图表 | 使用的字段 |
| --- | --- | --- |
| `monthly_chart` | 月度柱线组合图 | `order_month`、`valid_orders`、`item_gmv` |
| `category_chart` | 品类 Top N 条形图 | `category`、`item_gmv` / `item_count` / `orders` |
| `state_chart` | 州域散点图 | `customer_state`、`valid_orders`、`item_gmv`、`unique_customers` |
| `geo_chart` | 客户地理分布散点图 | `longitude`、`latitude`、`customers`、`item_gmv`、`zip_prefix` |
| `marketing_chart` | 渠道线索与转化率 | `origin`、`mql_count`、`closed_deals`、`conversion_rate` |
| `seller_3d` | 卖家三维散点图 | `valid_orders`、`item_gmv`、`freight_share`、`seller_id` |

`zoom()` 为图表统一加上内置缩放与滑块缩放，便于查看长尾数据。

In [ ]:
def _read_echarts_runtime():
    panel_spec = find_spec("panel")
    if panel_spec is None or not panel_spec.submodule_search_locations:
        raise RuntimeError("Anaconda 环境中未找到 Panel 自带的离线 ECharts 资源。")

    asset_root = Path(next(iter(panel_spec.submodule_search_locations))) / "dist" / "bundled" / "echarts"

    def read_one(pattern):
        matches = sorted(asset_root.glob(pattern))
        if not matches:
            raise FileNotFoundError(f"未找到离线图表资源：{pattern}")
        return matches[-1].read_text(encoding="utf-8")

    return read_one("echarts@*/dist/echarts.min.js"), read_one("echarts-gl@*/dist/echarts-gl.min.js")


_ECHARTS_JS, _ECHARTS_GL_JS = _read_echarts_runtime()


def display_pyechart(chart, slot="chart"):
    """将 Pyecharts 图表嵌入 Notebook 输出。"""
    html = chart.render_embed()
    external = re.compile(r'<script type="text/javascript" src="[^"]*echarts(?:-gl)?\.min\.js"></script>')
    assets = _ECHARTS_JS + (_ECHARTS_GL_JS if "echarts-gl" in chart.js_dependencies.items else "")
    html = external.sub(lambda _: f"<script>{assets}</script>", html)
    frame = html_lib.escape(html, quote=True)
    display(HTML(
        f'<iframe srcdoc="{frame}" style="width:100%;height:{chart.height};'
        'border:0;min-height:420px" loading="eager"></iframe>'
    ))

_WIDGET_OBSERVERS = globals().get("_WIDGET_OBSERVERS", {})

def replace_widget_observers(key, bindings):
    previous = _WIDGET_OBSERVERS.pop(key, [])
    for control, callback, names in previous:
        control.unobserve(callback, names=names)
    current = []
    for control, callback, names in bindings:
        control.observe(callback, names=names)
        current.append((control, callback, names))
    _WIDGET_OBSERVERS[key] = current

def zoom():
    return [opts.DataZoomOpts(type_="inside", range_start=0, range_end=100),
            opts.DataZoomOpts(type_="slider", range_start=0, range_end=100)]

def monthly_chart(frame, mode="combo"):
    x = frame["order_month"].tolist()
    orders = frame["valid_orders"].astype(int).tolist()
    gmv = frame["item_gmv"].round(2).tolist()
    if mode == "bar":
        chart = Bar().add_xaxis(x).add_yaxis("订单数", orders, label_opts=opts.LabelOpts(is_show=False))
        yaxis = opts.AxisOpts(name="订单数")
    elif mode == "line":
        chart = Line().add_xaxis(x).add_yaxis("商品 GMV", gmv, is_smooth=True, label_opts=opts.LabelOpts(is_show=False))
        yaxis = opts.AxisOpts(name="商品 GMV（BRL）")
    else:
        chart = Bar().add_xaxis(x).add_yaxis("订单数", orders, label_opts=opts.LabelOpts(is_show=False))
        chart.extend_axis(yaxis=opts.AxisOpts(name="商品 GMV（BRL）", position="right"))
        line = Line().add_xaxis(x).add_yaxis("商品 GMV", gmv, yaxis_index=1, is_smooth=True, label_opts=opts.LabelOpts(is_show=False))
        chart.overlap(line)
        yaxis = opts.AxisOpts(name="订单数")
    chart.set_global_opts(title_opts=opts.TitleOpts(title="月度订单与 GMV"), tooltip_opts=opts.TooltipOpts(trigger="axis"),
                          toolbox_opts=opts.ToolboxOpts(is_show=True), datazoom_opts=zoom(), yaxis_opts=yaxis,
                          xaxis_opts=opts.AxisOpts(name="月份", axislabel_opts=opts.LabelOpts(rotate=35)))
    return chart

def category_chart(frame, orientation="horizontal", metric="item_gmv", top_n=15):
    rows = frame.nlargest(top_n, metric)
    names, values = rows["category"].tolist(), rows[metric].round(2).tolist()
    metric_label = {"item_gmv": "商品 GMV（BRL）", "item_count": "商品件数", "orders": "订单数"}[metric]
    chart = Bar()
    if orientation == "horizontal":
        chart.add_xaxis(names[::-1]).add_yaxis(metric_label, values[::-1], label_opts=opts.LabelOpts(is_show=False)).reversal_axis()
        xaxis, yaxis = opts.AxisOpts(name=metric_label, type_="value"), opts.AxisOpts(name="商品品类")
    else:
        chart.add_xaxis(names).add_yaxis(metric_label, values, label_opts=opts.LabelOpts(is_show=False))
        xaxis, yaxis = opts.AxisOpts(name="商品品类", axislabel_opts=opts.LabelOpts(rotate=35)), opts.AxisOpts(name=metric_label)
    chart.set_global_opts(title_opts=opts.TitleOpts(title=f"商品品类 Top {top_n}：{metric_label}"),
                          tooltip_opts=opts.TooltipOpts(trigger="axis"), toolbox_opts=opts.ToolboxOpts(is_show=True),
                          datazoom_opts=zoom(), xaxis_opts=xaxis, yaxis_opts=yaxis)
    return chart

def state_chart(frame):
    points = [[int(row.valid_orders), round(row.item_gmv, 2), int(row.unique_customers), row.customer_state]
              for row in frame.itertuples()]
    chart = Scatter().add_xaxis([]).add_yaxis("州域", points, symbol_size=18, label_opts=opts.LabelOpts(is_show=False))
    chart.set_global_opts(title_opts=opts.TitleOpts(title="各州 GMV 与客户规模"),
                          tooltip_opts=opts.TooltipOpts(formatter="州域：{@[3]}<br/>订单数：{@[0]}<br/>商品 GMV：{@[1]}<br/>客户数：{@[2]}"),
                          toolbox_opts=opts.ToolboxOpts(is_show=True), datazoom_opts=zoom(),
                          xaxis_opts=opts.AxisOpts(name="订单数", type_="value"),
                          yaxis_opts=opts.AxisOpts(name="商品 GMV（BRL）", type_="value"))
    return chart

def geo_chart(frame, top_n=1500):
    """客户地理分布散点图（经纬度），气泡大小代表客户数。"""
    rows = frame.dropna(subset=["latitude", "longitude"]).nlargest(top_n, "customers")
    if rows.empty:
        raise ValueError("地理数据为空：请确认 geolocation 表已导入，且客户邮编能与 geolocation 匹配。")
    x = rows["longitude"].round(5).tolist()
    y = rows["latitude"].round(5).tolist()
    points = [[round(row.longitude, 5), round(row.latitude, 5), int(row.customers), round(row.item_gmv, 2), row.zip_prefix]
              for row in rows.itertuples()]
    chart = Scatter(init_opts=opts.InitOpts(width="100%", height="620px", bg_color="#0f1b2d"))
    chart.add_xaxis(x)
    chart.add_yaxis(
        "客户分布", points,
        symbol_size=7, label_opts=opts.LabelOpts(is_show=False),
        itemstyle_opts=opts.ItemStyleOpts(opacity=0.65, border_color="#7fb3ff", border_width=0.4),
    )
    chart.set_global_opts(
        title_opts=opts.TitleOpts(title=f"客户地理分布（客户数 Top {top_n} 邮编区）",
                                  title_textstyle_opts=opts.TextStyleOpts(color="#e8f1ff")),
        tooltip_opts=opts.TooltipOpts(
            trigger="item",
            formatter="邮编：{@[4]}<br/>客户数：{@[2]}<br/>商品 GMV：{@[3]}",
        ),
        toolbox_opts=opts.ToolboxOpts(is_show=True),
        visualmap_opts=opts.VisualMapOpts(
            type_="size", min_=int(rows["customers"].min()), max_=int(rows["customers"].max()),
            pos_right=10, pos_bottom=40, is_calculable=True, dimension=2,
        ),
        xaxis_opts=opts.AxisOpts(name="经度", type_="value",
                                 axislabel_opts=opts.LabelOpts(color="#9fb6d4"),
                                 splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(opacity=0.12))),
        yaxis_opts=opts.AxisOpts(name="纬度", type_="value",
                                 axislabel_opts=opts.LabelOpts(color="#9fb6d4"),
                                 splitline_opts=opts.SplitLineOpts(is_show=True, linestyle_opts=opts.LineStyleOpts(opacity=0.12))),
    )
    return chart

def marketing_chart(frame, mode="combo", minimum_mql=30):
    rows = frame[frame["mql_count"] >= minimum_mql].sort_values("mql_count", ascending=False)
    x = rows["origin"].tolist()
    mql = rows["mql_count"].astype(int).tolist()
    closed = rows["closed_deals"].astype(int).tolist()
    rate = (rows["conversion_rate"] * 100).round(2).tolist()
    if mode == "line":
        chart = Line().add_xaxis(x).add_yaxis("转化率（%）", rate, is_smooth=True, label_opts=opts.LabelOpts(is_show=False))
    else:
        chart = Bar().add_xaxis(x).add_yaxis("营销线索数", mql, label_opts=opts.LabelOpts(is_show=False)).add_yaxis("成交客户数", closed, label_opts=opts.LabelOpts(is_show=False))
        if mode == "combo":
            chart.extend_axis(yaxis=opts.AxisOpts(name="转化率（%）", position="right"))
            chart.overlap(Line().add_xaxis(x).add_yaxis("转化率（%）", rate, yaxis_index=1, is_smooth=True, label_opts=opts.LabelOpts(is_show=False)))
    chart.set_global_opts(title_opts=opts.TitleOpts(title="营销渠道转化率"),
                          tooltip_opts=opts.TooltipOpts(trigger="axis"), toolbox_opts=opts.ToolboxOpts(is_show=True),
                          datazoom_opts=zoom(), xaxis_opts=opts.AxisOpts(name="营销渠道", axislabel_opts=opts.LabelOpts(rotate=25)))
    return chart

def seller_3d(frame, top_n=200):
    rows = frame.nlargest(top_n, "item_gmv")
    values = [[int(row.valid_orders), round(row.item_gmv, 2), round(float(row.freight_share or 0), 4), row.seller_id]
              for row in rows.itertuples()]
    chart = Scatter3D(init_opts=opts.InitOpts(width="1100px", height="700px"))
    chart.add("卖家", values, xaxis3d_opts=opts.Axis3DOpts(name="订单数"),
              yaxis3d_opts=opts.Axis3DOpts(name="商品 GMV（BRL）"), zaxis3d_opts=opts.Axis3DOpts(name="运费占比"),
              label_opts=opts.LabelOpts(is_show=False))
    chart.set_global_opts(title_opts=opts.TitleOpts(title="卖家经营表现 3D 分析"),
                          tooltip_opts=opts.TooltipOpts(formatter="卖家：{@[3]}<br/>订单数：{@[0]}<br/>商品 GMV：{@[1]}<br/>运费占比：{@[2]}"))
    return chart

def late_rate_chart(frame, min_orders=300, top_n=10):
    """各州迟到率与平均履约天数（仅纳入订单量达标的州）。"""
    rows = frame[frame["orders"] >= min_orders].nlargest(top_n, "late_rate")
    x = rows["customer_state"].tolist()
    rate = (rows["late_rate"] * 100).round(2).tolist()
    days = rows["avg_delivery_days"].round(1).tolist()
    chart = Bar(init_opts=opts.InitOpts(width="100%", height="460px"))
    chart.add_xaxis(x)
    chart.add_yaxis("迟到率（%）", rate, label_opts=opts.LabelOpts(is_show=True, position="top"))
    chart.extend_axis(yaxis=opts.AxisOpts(name="平均履约天数", position="right"))
    chart.overlap(Line().add_xaxis(x).add_yaxis("平均履约天数", days, yaxis_index=1,
                 is_smooth=True, label_opts=opts.LabelOpts(is_show=False)))
    chart.set_global_opts(
        title_opts=opts.TitleOpts(title=f"各州迟到率（仅统计订单数 ≥ {min_orders} 的州）"),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        toolbox_opts=opts.ToolboxOpts(is_show=True),
        xaxis_opts=opts.AxisOpts(name="客户所在州", axislabel_opts=opts.LabelOpts(rotate=0)),
        yaxis_opts=opts.AxisOpts(name="迟到率（%）"),
    )
    return chart


def delay_bucket_chart(frame):
    """延迟天数分桶 vs 平均评分 / 差评率。"""
    x = frame["bucket"].astype(str).tolist()
    score = frame["avg_score"].round(3).tolist()
    bad = (frame["bad_review_rate"] * 100).round(1).tolist()
    chart = Bar(init_opts=opts.InitOpts(width="100%", height="460px"))
    chart.add_xaxis(x)
    chart.add_yaxis("差评率（≤2星，%）", bad, label_opts=opts.LabelOpts(is_show=True, position="top"))
    chart.extend_axis(yaxis=opts.AxisOpts(name="平均评分", position="right", min_=0, max_=5))
    chart.overlap(Line().add_xaxis(x).add_yaxis("平均评分", score, yaxis_index=1,
                 is_smooth=True, label_opts=opts.LabelOpts(is_show=True, position="top")))
    chart.set_global_opts(
        title_opts=opts.TitleOpts(title="延迟天数与评分：迟到超过 4 天后差评见顶"),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        toolbox_opts=opts.ToolboxOpts(is_show=True),
        xaxis_opts=opts.AxisOpts(name="相对预计送达时间"),
        yaxis_opts=opts.AxisOpts(name="差评率（%）", max_=100),
    )
    return chart

## 6. 月度趋势：GMV / 订单 / 客户

基于 `ANALYSIS["monthly"]`（24 个月）观察经营规模走势。

- 组合图：柱形为 `valid_orders`（订单数），折线为 `item_gmv`（商品销售额），双轴；
- 两个交互控件：`month_mode` 切换图表形态，`month_range` 选取月份区间；
- 图表随控件实时重绘。

> 首尾月份订单量明显偏低，属于数据集边界效应（首月与末月并非完整自然月）。
> 解读趋势时应剔除或单独说明，不要把断崖当作业务下滑。

In [ ]:
month_mode = widgets.ToggleButtons(options=[("组合图", "combo"), ("柱状图", "bar"), ("折线图", "line")], description="图表")
month_range = widgets.SelectionRangeSlider(options=ANALYSIS["monthly"]["order_month"].tolist(),
    index=(0, len(ANALYSIS["monthly"]) - 1), description="月份区间", continuous_update=False,
    layout=widgets.Layout(width="760px"))
month_output = widgets.Output()

def show_monthly(change=None):
    start, end = month_range.value
    frame = ANALYSIS["monthly"].query("@start <= order_month <= @end")
    with month_output:
        clear_output(wait=True)
        display_pyechart(monthly_chart(frame, month_mode.value), "monthly")

replace_widget_observers("monthly", [(month_mode, show_monthly, "value"), (month_range, show_monthly, "value")])
ui = widgets.VBox([month_mode, month_range])
display(ui)
display(month_output)
show_monthly()

## 7. 商品品类分析

基于 `ANALYSIS["category"]`，按品类汇总销售额与销量。

品类名取自 `product_category_name_translation.product_category_name_english`；
未匹配到译名的商品归入 `unknown`。

三个交互控件：

- `category_orientation`：横向 / 纵向；
- `category_metric`：指标三选一——`item_gmv`（商品销售额）、
  `item_count`（商品件数）、`orders`（订单数）；
- `category_top`：显示前 N 个品类。

三者口径不同：`item_gmv` 累计金额，`item_count` 数商品件数，`orders` 数去重订单数。
同一品类在三者下的排名可能不一致——单价高的品类金额靠前，件数未必靠前。

In [ ]:
category_orientation = widgets.ToggleButtons(options=[("横向", "horizontal"), ("纵向", "vertical")], description="图表")
category_metric = widgets.Dropdown(options=[("货品销售总额", "item_gmv"),("单品销售额", "item_count"),("订单数", "orders")], value="item_gmv", description="指标")
category_top = widgets.IntSlider(value=PARAMS["category_top_n"], min=5, max=50, step=5, description="Top N", continuous_update=False)
category_output = widgets.Output()

def show_category(change=None):
    with category_output:
        clear_output(wait=True)
        display_pyechart(category_chart(ANALYSIS["category"], category_orientation.value, category_metric.value, category_top.value), "category")

replace_widget_observers("category", [(control, show_category, "value") for control in [category_orientation, category_metric, category_top]])
display(widgets.VBox([category_orientation, category_metric, category_top]))
display(category_output)
show_category()

## 8. 州域、客户地理分布与营销渠道

**州域散点图**（`ANALYSIS["state"]`）：横轴 `valid_orders`，纵轴 `item_gmv`，
提示框额外展示 `unique_customers`，用于观察各州规模与结构差异。

**客户地理分布**（`ANALYSIS["geo_points"]`）：把 `customer_zip_code_prefix`
与 `geolocation` 按邮编关联，取到该邮编区的 `latitude` / `longitude`，
绘制经纬度散点，气泡大小对应客户数 `customers`。
邮编在源数据中会重复（`geolocation` 表无主键），已在 SQL 侧按邮编取平均经纬度去重。
默认展示客户数最多的 1200 个邮编区。

**营销渠道**（`ANALYSIS["marketing"]`）：

- `mql_count`：该 `origin` 的营销合格线索数（按 `mql_id` 去重）；
- `closed_deals`：其中标记为已成交的线索数；
- `conversion_rate` = `closed_deals` ÷ `mql_count`；
- `minimum_mql` 滑块过滤线索量过少的渠道，避免小样本导致转化率虚高。

> 营销数据与交易数据之间没有稳定的用户级关联键，因此这里的转化率只描述线索流转，
> 不能用于广告 ROI、用户级归因或因果推断。

In [ ]:
display_pyechart(state_chart(ANALYSIS["state"]), "state")

display(Markdown("### 客户地理分布"))
display_pyechart(geo_chart(ANALYSIS["geo_points"], top_n=1200), "geo")

marketing_mode = widgets.ToggleButtons(options=[("组合", "combo"), ("柱状图", "bar"), ("折线图", "line")], description="展示方式")
minimum_mql = widgets.IntSlider(value=PARAMS["minimum_mql"], min=0, max=500, step=10, description="最小线索数", continuous_update=False)
marketing_output = widgets.Output()

def show_marketing(change=None):
    with marketing_output:
        clear_output(wait=True)
        display_pyechart(marketing_chart(ANALYSIS["marketing"], marketing_mode.value, minimum_mql.value), "marketing")

replace_widget_observers("marketing", [(marketing_mode, show_marketing, "value"), (minimum_mql, show_marketing, "value")])
display(widgets.VBox([marketing_mode, minimum_mql]))
display(marketing_output)
show_marketing()

## 9. 履约分析与评分归因

这一节回答一个业务问题：**配送延迟到底值多少钱？** 结论是可以量化的。

分析基于 `ANALYSIS["delivery_buckets"]`、`ANALYSIS["delivery_state"]` 和
`ANALYSIS["delivery_lane"]`，口径如下：

- 只纳入`order_delivered_customer_date` 与 `order_estimated_delivery_date` 都非空的订单；
- `days_late` = 实际送达 − 预计送达（负数截断为 0），`is_late` = `days_late > 0`；
- `delivery_days` = 实际送达 − 下单时间，衡量端到端履约时长；
- 评分取自 `order_reviews.review_score`（一单多条评价时取均值）。

**核心发现是评分对延迟呈非线性下降**：迟到 1–3 天已经开始明显掉分，
超过 4 天则断崖式下跌，但再继续恶化时差评率会见顶而不再上升。
这意味着「把延迟从 10 天压到 5 天」的收益，远大于「从 3 天压到 0 天」以外的任何边际改善。

In [ ]:
from IPython.display import display as _display

_timely = ANALYSIS["orders"].dropna(
    subset=["order_delivered_customer_date", "order_estimated_delivery_date"])
_late = _timely["is_late"]

display(Markdown("### 总体履约情况"))
display(pd.Series({
    "可评估订单数": int(len(_timely)),
    "迟到订单数": int(_late.sum()),
    "迟到率": _late.mean(),
    "平均履约天数": _timely["order_delivered_customer_date"].sub(
        _timely["order_purchase_timestamp"]).dt.days.mean(),
    "平均提前天数": _timely["order_estimated_delivery_date"].sub(
        _timely["order_delivered_customer_date"]).dt.days.mean(),
}).to_frame("值"))

display(Markdown("### 延迟天数分桶：评分与差评率"))
_buckets = ANALYSIS["delivery_buckets"].copy()
_buckets["avg_score"] = _buckets["avg_score"].round(3)
_buckets["bad_review_rate"] = _buckets["bad_review_rate"].map("{:.2%}".format)
display(_buckets.rename(columns={
    "bucket": "延迟分桶", "orders": "订单数",
    "avg_score": "平均评分", "bad_review_rate": "差评率（≤2星）"}))

display_pyechart(delay_bucket_chart(ANALYSIS["delivery_buckets"]), "delay")

display(Markdown("### 各州迟到率"))
display_pyechart(late_rate_chart(ANALYSIS["delivery_state"]), "state_late")

display(Markdown("### 同州 vs 跨州"))
_lane = ANALYSIS["delivery_lane"].copy()
_lane["late_rate"] = _lane["late_rate"].map("{:.2%}".format)
_lane["avg_freight"] = _lane["avg_freight"].round(2)
display(_lane.rename(columns={
    "lane": "履约路径", "lines": "明细数", "late_orders": "迟到明细数",
    "late_rate": "迟到率", "avg_freight": "平均运费（BRL）"}))

## 10. 中文字体与 Matplotlib 导出

Matplotlib 默认字体不含中日韩字形，中文会渲染成方框，因此先配置字体再画图。

`configure_matplotlib_chinese()` 的做法：按「字体族名 + 文件名」候选表在
`C:\Windows\Fonts` 与用户字体目录中探测真实文件，用
`FontProperties(fname=...).get_name()` 取回可用族名后写入
`plt.rcParams["font.sans-serif"]`。

> 注意：不要拿 `findSystemFonts()` 的返回值和族名直接比较——它返回的是**文件路径**
> （如 `msyh.ttc`），与 `Microsoft YaHei` 永不相等，会导致选字体恒失败。
> 另外，只有在成功选中中文字体后才屏蔽缺字告警，失败时仍会抛出提醒。

本单元格同时渲染卖家三维图（`seller_3d`）。三维图依赖 echarts-gl，首次渲染稍慢；
末尾插入一段留白，避免 iframe 与静态画布贴在一起。

In [ ]:
from matplotlib import font_manager
import warnings

def configure_matplotlib_chinese():
    """Select an installed CJK font so Chinese labels render correctly."""
    # 字体族名 -> 常见文件名。直接探测文件可避免依赖 findSystemFonts() 的
    # 平台差异：findSystemFonts() 返回的是文件路径，不能再和字体族名比较。
    candidates = [
        ("Microsoft YaHei", "msyh.ttc"),
        ("Microsoft YaHei", "msyh.ttf"),
        ("Microsoft YaHei UI", "msyh.ttc"),
        ("DengXian", "Deng.ttf"),
        ("SimHei", "simhei.ttf"),
        ("Noto Sans CJK SC", "NotoSansCJK-Regular.ttc"),
        ("Source Han Sans CN", "SourceHanSansCN-Regular.otf"),
        ("WenQuanYi Zen Hei", "wqy-zenhei.ttc"),
        ("Arial Unicode MS", "arialuni.ttf"),
    ]

    font_dirs = [Path(r"C:\Windows\Fonts"), Path.home() / "AppData" / "Local" / "Microsoft" / "Windows" / "Fonts"]
    selected, matched_font = None, None
    for name, filename in candidates:
        for directory in font_dirs:
            font_path = directory / filename
            if not font_path.is_file():
                continue
            try:
                family = font_manager.FontProperties(fname=str(font_path)).get_name()
            except Exception:
                continue
            selected, matched_font = (family or name), font_path
            break
        if selected:
            break

    plt.rcParams["axes.unicode_minus"] = False
    if selected:
        plt.rcParams["font.family"] = "sans-serif"
        plt.rcParams["font.sans-serif"] = [selected, "DejaVu Sans"]
        # 已选定中文字体，个别生僻字缺字时不再刷屏告警
        warnings.filterwarnings("ignore", message=".*Glyph .* missing from font.*")
    else:
        warnings.warn(
            "未找到可用中文字体，中文标签可能显示为方框；"
            "请在系统中安装 Microsoft YaHei、SimHei 或 Noto Sans CJK SC。",
            RuntimeWarning,
        )
    return selected, matched_font

MATPLOTLIB_FONT, MATPLOTLIB_FONT_FILE = configure_matplotlib_chinese()
display(Markdown("### 卖家 3D 交互图"))
display_pyechart(seller_3d(ANALYSIS["seller"], PARAMS["seller_top_n"]), "seller")

# 用独立的 Notebook 输出和显式留白，避免 3D iframe 与静态画布连在一起。
display(HTML('<div style="height:32px"></div>'))


### Matplotlib 静态图

把月度商品销售额画成折线并导出 PNG，用于报告与项目展示。

- 数据源：`ANALYSIS["monthly"]`；
- 导出路径：`notebooks/exports/monthly_item_gmv.png`（目录不存在会自动创建）；
- 末尾打印实际生效的中文字体名与字体文件路径，便于排查渲染问题。

In [ ]:
display(Markdown("### 月度 GMV 静态图（Matplotlib）"))

monthly = ANALYSIS["monthly"]
fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
ax.plot(
    monthly["order_month"],
    monthly["item_gmv"],
    marker="o",
    linewidth=2,
    color="#2f6f9f",
)
ax.set(
    title="月度商品 GMV",
    xlabel="月份",
    ylabel="商品 GMV（BRL）",
)
ax.tick_params(axis="x", rotation=45)
ax.grid(alpha=0.25)
ax.margins(x=0.02)

EXPORT_DIR = ROOT / "notebooks" / "exports"
EXPORT_DIR.mkdir(exist_ok=True)
STATIC_EXPORT = EXPORT_DIR / "monthly_item_gmv.png"
fig.savefig(STATIC_EXPORT, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"中文字体：{MATPLOTLIB_FONT or '未找到专用中文字体'}")
print(f"字体文件：{MATPLOTLIB_FONT_FILE or '未匹配到已知字体文件'}")
print(f"静态图已导出：{STATIC_EXPORT}")


## 11. 客户 RFM 分群

客户粒度统一使用 `customer_unique_id`（`customer_id` 会随订单变化，不能代表同一个人）。

### 三个原始指标

在 `build_analysis` 中计算：

| 指标 | 字段 | 定义 |
| --- | --- | --- |
| 最近购买间隔 | `recency` | 分析截止日 − `last_purchase_date`，单位为天 |
| 购买频次 | `frequency` | 该客户的去重订单数 |
| 消费金额 | `monetary` | 该客户累计 `payment_value` |

分析截止日取有效订单最大 `order_purchase_timestamp` 再加 1 天（`snapshot_date`），
避免最近下单的客户 `recency` 为 0 而落到边界之外。

### 打分与分群

`quintile_score()` 用 `pd.qcut` 做五分位；`r_score` 为反向评分，
`recency` 越小（越近）得分越高。三个分数拼成 `rfm_score`。

`segment` 按优先级依次判定，先命中先归类：
`Champions`（核心客户）→ `Loyal Customers`（忠诚客户）→ `Big Spenders`（高消费客户）
→ `Recent Customers`（新近客户）→ `At Risk`（流失风险客户）→ `Others`（其他客户）。

### 产出

`rfm_summary`（各分群的客户数、平均 R/F、消费金额与占比）与 `rfm` 明细
（按 R/F/M 得分降序取前 20 名）。单元格末尾以断言校验客户唯一性、
`recency ≥ 1`、分群名称合法，以及汇总客户数与金额能和明细对上。

In [ ]:
SEGMENT_LABELS = {
    "Champions": "核心客户",
    "Loyal Customers": "忠诚客户",
    "Big Spenders": "高消费客户",
    "Recent Customers": "新近客户",
    "At Risk": "流失风险客户",
    "Others": "其他客户",
}

rfm_summary = ANALYSIS["rfm_summary"].copy()
rfm_summary["segment"] = rfm_summary["segment"].astype(str)
rfm_summary_display = rfm_summary.rename(columns={
    "segment": "客户分群", "customers": "客户数", "avg_recency": "平均最近购买间隔（天）",
    "avg_frequency": "平均购买频次", "revenue": "消费金额（BRL）", "customer_share": "客户占比", "revenue_share": "消费金额占比",
}).copy()
rfm_summary_display["客户分群"] = rfm_summary_display["客户分群"].map(SEGMENT_LABELS).fillna(rfm_summary_display["客户分群"])
display(Markdown("### 客户 RFM 分群汇总"))
display(rfm_summary_display.style.format({"平均最近购买间隔（天）": "{:.1f}", "平均购买频次": "{:.2f}", "消费金额（BRL）": "{:,.2f}", "客户占比": "{:.2%}", "消费金额占比": "{:.2%}"}))
display(Markdown("### 客户 RFM 明细（前 20 名）"))
rfm_detail_display = ANALYSIS["rfm"].sort_values(["r_score", "f_score", "m_score", "monetary"], ascending=[False, False, False, False]).head(20).rename(columns={
    "customer_unique_id": "客户唯一标识", "last_purchase_date": "最近购买日期", "frequency": "购买频次", "monetary": "消费金额（BRL）",
    "recency": "最近购买间隔（天）", "r_score": "R 得分", "f_score": "F 得分", "m_score": "M 得分", "rfm_score": "RFM 得分", "segment": "客户分群",
})
rfm_detail_display["客户分群"] = rfm_detail_display["客户分群"].map(SEGMENT_LABELS).fillna(rfm_detail_display["客户分群"])
display(rfm_detail_display)
assert ANALYSIS["rfm"]["customer_unique_id"].is_unique
assert ANALYSIS["rfm"]["recency"].ge(1).all()
assert set(ANALYSIS["rfm"]["segment"]) <= set(SEGMENT_LABELS)
assert rfm_summary["customers"].sum() == len(ANALYSIS["rfm"])
assert abs(rfm_summary["revenue"].sum() - ANALYSIS["rfm"]["monetary"].sum()) < 1e-6
print(f"RFM 客户数：{len(ANALYSIS['rfm']):,}，客户分群数：{rfm_summary['segment'].nunique()}。")

## 12. 业务结论

把 `ANALYSIS` 中分散的结果收拢成可直接引用的结论：

- 商品销售额最高的月份（`monthly` 中 `item_gmv` 最大的行）；
- 销售额最高的商品品类（`category`）；
- 销售额最高的州域（`state`）；
- 客户数最多的 RFM 分群（`rfm_summary`）；
- 线索量达标（`minimum_mql` 以上）的渠道中转化率最高者（`marketing`）。

每条结论均由数据实时计算，改动前面的 `PARAMS` 或清洗规则后重跑本节即可同步更新。
解读时请结合 README「项目限制」中的口径说明与数据边界（公开历史数据、
部分订单缺评价或日期、营销数据无法与交易数据做用户级关联）。

In [ ]:
monthly = ANALYSIS["monthly"]
category = ANALYSIS["category"]
state = ANALYSIS["state"]
marketing = ANALYSIS["marketing"]
rfm_summary = ANALYSIS["rfm_summary"]
top_month = monthly.loc[monthly["item_gmv"].idxmax()]
top_category = category.loc[category["item_gmv"].idxmax()]
top_state = state.loc[state["item_gmv"].idxmax()]
eligible = marketing[marketing["mql_count"] >= PARAMS["minimum_mql"]]
best_origin = eligible.loc[eligible["conversion_rate"].idxmax()]
largest_segment = rfm_summary.loc[rfm_summary["customers"].idxmax()]
largest_segment_name = SEGMENT_LABELS.get(largest_segment["segment"], largest_segment["segment"])
display(Markdown("\n".join([
    f"- GMV 最高月份：**{top_month['order_month']}**，GMV 为 **{top_month['item_gmv']:,.2f}**。",
    f"- GMV 最高的商品品类：**{top_category['category']}**，GMV 为 **{top_category['item_gmv']:,.2f}**。",
    f"- GMV 最高的州域：**{top_state['customer_state']}**，GMV 为 **{top_state['item_gmv']:,.2f}**。",
    f"- 客户数最多的 RFM 分群：**{largest_segment_name}**，共有 **{int(largest_segment['customers']):,}** 位客户（{largest_segment['customer_share']:.2%}）。",
    f"- 在不少于 {PARAMS['minimum_mql']} 条营销线索的渠道中，转化率最高的是 **{best_origin['origin']}**，转化率为 **{best_origin['conversion_rate']:.2%}**。",
    "- 结合订单、客户、营销和地理数据，为运营策略与 ROI 评估提供依据。",
])))